# Loan Approval - EDA

Working through the Analytics Vidhya loan prediction dataset.
Goals:
1. data shape, dtypes
2. missing values
3. target balance
4. univariate plots
5. bivariate vs target
6. correlations
7. quick model comparison

## Findings (will write up in README)
- About 31% of values in Credit_History column are missing/zero matters a lot
- Target is imbalanced (~69% Y, 31% N), worth checking F1 not just accuracy
- Income features are very right-skewed, log helps
- Credit_History is the dominant predictor


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set(style='whitegrid')

In [ ]:
df = pd.read_csv('../data/train.csv')
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## Missing values
Credit_History, Self_Employed, LoanAmount have the most missing values.

In [ ]:
miss = df.isnull().sum().sort_values(ascending=False)
miss = miss[miss > 0]
miss

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
miss.plot.bar(ax=ax, color='salmon')
ax.set_ylabel('# missing')
ax.set_title('Missing values per column')
plt.tight_layout()

## Target balance
Loan_Status: Y means approved, N means rejected.
About 69% are approved, so a baseline that always predicts Y already gets ~69% accuracy.

In [ ]:
df['Loan_Status'].value_counts(normalize=True)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(x='Loan_Status', data=df, ax=ax)
ax.set_title('Loan_Status distribution')

## Categorical features vs target

In [ ]:
cat_cols = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area', 'Credit_History']
fig, axes = plt.subplots(3, 3, figsize=(13, 10))
axes = axes.flatten()
for i, c in enumerate(cat_cols):
    sns.countplot(x=c, hue='Loan_Status', data=df, ax=axes[i])
    axes[i].set_title(c)
for j in range(len(cat_cols), len(axes)):
    axes[j].axis('off')
plt.tight_layout()

## Numeric distributions
Income looks heavily skewed. Will need a log transform if we use a linear model.
Note: drop NaN before plotting hist (matplotlib will warn otherwise).

In [ ]:
num_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, c in zip(axes.flatten(), num_cols):
    df[c].dropna().plot.hist(bins=40, ax=ax, color='steelblue')
    ax.set_title(c)
plt.tight_layout()

In [ ]:
df['log_ApplicantIncome'] = np.log1p(df['ApplicantIncome'])
fig, ax = plt.subplots(figsize=(7, 4))
df['log_ApplicantIncome'].dropna().plot.hist(bins=40, ax=ax, color='seagreen')
ax.set_title('log(1 + ApplicantIncome)')

## Correlations
Credit_History stands out as the strongest predictor of approval.

In [ ]:
tmp = df.copy()
tmp['target'] = (tmp['Loan_Status'] == 'Y').astype(int)
corr = tmp[num_cols + ['Credit_History', 'target']].corr()
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Numeric correlations')

## Quick model comparison
Spot check three families: LogReg, RandomForest, GradientBoosting.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score

work = df.drop(columns=['Loan_ID', 'log_ApplicantIncome']).copy()
for c in ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']:
    work[c] = work[c].fillna(work[c].mode()[0])
for c in ['LoanAmount', 'Loan_Amount_Term', 'Credit_History']:
    work[c] = work[c].fillna(work[c].median())
for c in ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']:
    work[c] = LabelEncoder().fit_transform(work[c].astype(str))
y = (work['Loan_Status'] == 'Y').astype(int)
X = work.drop(columns=['Loan_Status'])

models = {
    'logreg': LogisticRegression(max_iter=1000),
    'rf': RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42),
    'gb': GradientBoostingClassifier(random_state=42),
}
for name, m in models.items():
    s = cross_val_score(m, X, y, cv=5, scoring='accuracy')
    print(name, 'mean acc =', round(s.mean(), 4), '+-', round(s.std(), 4))